# 02 · Exploratory Data Analysis

Goal: understand the cleaned dataset before any modeling.
All data comes from `data/processed/pokemon.csv`, built by `python -m src.data.build_dataset`.

In [ ]:
# Setup: let this notebook import from src/ (the notebook runs from notebooks/, src/ lives one level up)
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.executable)  # must end in pokelab\.venv\Scripts\python.exe

In [ ]:
import pandas as pd

from src.config import PROCESSED_DIR
from src.data.build_dataset import STAT_COLS  # same stat list the pipeline uses

df = pd.read_csv(PROCESSED_DIR / "pokemon.csv")
print(df.shape)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")

# One label per Pokémon: mythical > legendary > regular
df["category"] = np.select([df["is_mythical"], df["is_legendary"]], ["mythical", "legendary"], "regular")

## 1. What's in the dataset?

In [ ]:
print(df["category"].value_counts(), "\n")
print("Dual-typed:", df["type_2"].notna().sum(), "of", len(df))

# A Pokémon counts once for each of its types
type_counts = pd.concat([df["type_1"], df["type_2"].dropna()]).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["generation"].value_counts().sort_index().plot.bar(ax=axes[0], title="Species per generation")
type_counts.plot.bar(ax=axes[1], title="Pokémon per type (primary + secondary)")
plt.tight_layout()

## 2. How are the six stats distributed?

Skew > 0 means a long right tail: a few Pokémon with very high values. Outliers matter because StandardScaler uses the mean and standard deviation, which outliers pull around.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, STAT_COLS):
    sns.histplot(df[col], bins=30, ax=ax)
    ax.set_title(f"{col}  (skew = {df[col].skew():.2f})")
plt.tight_layout()

df[STAT_COLS + ["bst"]].describe().round(1)

In [ ]:
# The most extreme Pokémon for each stat: are these real, or data problems?
for col in STAT_COLS:
    top = df.nlargest(3, col)[["name", col]].values.tolist()
    low = df.nsmallest(3, col)[["name", col]].values.tolist()
    print(f"{col:11s} high: {top}\n{'':11s} low:  {low}")

## 3. Base stat total (BST): shape, legendaries, generations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["bst"], bins=40, ax=axes[0])
axes[0].set_title("BST distribution: how many peaks?")

sns.boxplot(data=df, x="category", y="bst", order=["regular", "legendary", "mythical"], ax=axes[1])
axes[1].set_title("BST by category")

sns.boxplot(data=df[df["category"] == "regular"], x="generation", y="bst", ax=axes[2])
axes[2].set_title("BST by generation (regular Pokémon only)")
plt.tight_layout()

df.groupby("category")["bst"].describe().round(0)

## 4. How do the stats relate to each other?

Left: raw stats (what Representation A clusters on).
Right: relative stats, stat / BST (Representation B).

Relative stats always sum to 1, so if one share goes up, the others must go down on average. With 6 parts, the average correlation is forced toward **−1 / (6 − 1) = −0.2** even if nothing interesting is going on. Only correlations clearly below or above that baseline mean something.

In [ ]:
relative = df[STAT_COLS].div(df["bst"], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in [(axes[0], df[STAT_COLS], "Raw stats"), (axes[1], relative, "Relative stats (stat / BST)")]:
    sns.heatmap(data.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title(title)
plt.tight_layout()

off_diag = lambda c: c.where(~np.eye(len(c), dtype=bool)).stack()
print("Average off-diagonal correlation, raw:     ", round(off_diag(df[STAT_COLS].corr()).mean(), 2))
print("Average off-diagonal correlation, relative:", round(off_diag(relative.corr()).mean(), 2))
print("\nEach raw stat's correlation with BST:")
print(df[STAT_COLS].corrwith(df["bst"]).round(2))

## 5. Evolution stage: the hidden variable behind BST

If the two BST peaks are 'still evolving' vs 'fully evolved', then any comparison across groups must check whether the groups have different shares of unevolved Pokémon.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df, x="bst", hue="is_fully_evolved", bins=40, multiple="stack", ax=axes[0])
axes[0].set_title("BST split by fully evolved")

regular = df[df["category"] == "regular"]
by_gen = pd.DataFrame({
    "all regular": regular.groupby("generation")["bst"].median(),
    "fully evolved only": regular[regular["is_fully_evolved"]].groupby("generation")["bst"].median(),
})
by_gen.plot(marker="o", ax=axes[1], title="Median BST by generation")
plt.tight_layout()

print("Share NOT fully evolved, by generation:")
print((1 - df.groupby("generation")["is_fully_evolved"].mean()).round(2).to_dict())
by_gen